In [8]:
import os

import ee
import geemap

from find_set_root import find_set_project_root
PROJECT_ROOT = find_set_project_root()
print(f"Project root found at: {PROJECT_ROOT}")
import utils.general_functions as ugf


DIR_RAW = os.path.join(PROJECT_ROOT, 'data', 'raw')
DIR_DERIVED = os.path.join(PROJECT_ROOT, 'data', 'derived')
ugf.dir_ensure([DIR_RAW, DIR_DERIVED])

GEE_PROJECT = 'ee-tymc5571-multi-disturbance'
GEE_PROJECT_DIR = 'projects/ee-tymc5571-multi-disturbance/assets'
DRIVE_FOLDER = "GEE_Exports_resilience_20260212"


#Prepare to use Earth Engine
ee.Authenticate(
    auth_mode='notebook',
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/devstorage.read_write',
        'https://www.googleapis.com/auth/drive'
    ]
    #, force = True
)
ee.Initialize(project=GEE_PROJECT)

SERVICE_FILE = "C:/Users/tymc5571/dev/cbi-module/config/secrets/tymc5571-utils-project-692f27c034dd.json"  # Path to service account file

EXPORT_CRS = 'EPSG:5070'


Project root found at: C:\Users\tymc5571\dev\compound-disturbance-resilience
✅ Directory already exists: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\raw
✅ Directory already exists: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\derived


In [9]:
biotic = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/biotic_gridded_1km_all_years_severity")
relaxed_forest = ee.Image("projects/ee-tymc5571-cbi-module/assets/relaxed_forest_1985_2024")

states = ee.FeatureCollection("TIGER/2018/States")
western_states_list = ['WA','OR','CA','ID','NV','MT','WY','UT','CO','AZ','NM']
western_states = states.filter(
    ee.Filter.inList('STUSPS', western_states_list)
)

In [26]:
relaxed_forest.bandNames().getInfo()

['forest_1985',
 'forest_1986',
 'forest_1987',
 'forest_1988',
 'forest_1989',
 'forest_1990',
 'forest_1991',
 'forest_1992',
 'forest_1993',
 'forest_1994',
 'forest_1995',
 'forest_1996',
 'forest_1997',
 'forest_1998',
 'forest_1999',
 'forest_2000',
 'forest_2001',
 'forest_2002',
 'forest_2003',
 'forest_2004',
 'forest_2005',
 'forest_2006',
 'forest_2007',
 'forest_2008',
 'forest_2009',
 'forest_2010',
 'forest_2011',
 'forest_2012',
 'forest_2013',
 'forest_2014',
 'forest_2015',
 'forest_2016',
 'forest_2017',
 'forest_2018',
 'forest_2019',
 'forest_2020',
 'forest_2021',
 'forest_2022',
 'forest_2023',
 'forest_2024']

In [3]:
def remove_to_bands_append(image):
    """
    Renames bands in the input Earth Engine Image by removing the first part
    (before the first underscore) from each band name.

    Args:
        image (ee.Image): The input image.

    Returns:
        ee.Image: The image with renamed bands.
    """
    old_names = image.bandNames()
    new_names = old_names.map(lambda name: ee.String(name).split('_').slice(1).join('_'))
    return image.rename(new_names)

In [27]:
#########################
# PREP BIOTIC DATA
#########################

new_biotic_names = [f'biotic_{year}' for year in range(1997, 2024)]
biotic = biotic.rename(new_biotic_names)


biotic_timeseries_sum = biotic.reduce(ee.Reducer.sum()).rename('biotic_timeseries_sum')
biotic_with_sum = biotic.addBands(biotic_timeseries_sum)

###########################
# Normalize by forested area
###########################

biotic_years = biotic.bandNames().map(
    lambda b: ee.Number.parse(ee.String(b).split('_').get(-1))
)

BIOTIC_FIRST_YEAR = ee.Number(biotic_years.sort().get(0))
BIOTIC_LAST_YEAR  = ee.Number(biotic_years.sort().get(-1))

def forest_fraction_prior_on_biotic_grid(year: ee.Number, *, max_pixels=4096) -> ee.Image:
    """Forest fraction (0..1) for (year-1) on the native biotic grid."""
    prior = ee.Number(year).subtract(1).format('%d')

    forest_30m = (relaxed_forest
        .select(ee.String("forest_").cat(prior))
        .unmask(0)
        .toFloat()
    )

    forest_frac = (forest_30m
    .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=max_pixels)
    .rename(ee.String("forestfrac_").cat(prior))
    )

    return forest_frac

def biotic_forestnorm_one_year(year: ee.Number) -> ee.Image:
    y_str = ee.Number(year).format('%d')

    biotic_y = (biotic
        .select(ee.String("biotic_").cat(y_str))
        .toFloat()
    )

    forest_frac_prior = forest_fraction_prior_on_biotic_grid(year)

    forest_frac_prior = forest_frac_prior.updateMask(forest_frac_prior.gt(0))

    norm = (biotic_y
        .divide(forest_frac_prior)
        .min(100)
        .rename(ee.String("biotic_forestnorm_").cat(y_str))
    )

    return norm

years_list = ee.List.sequence(BIOTIC_FIRST_YEAR, BIOTIC_LAST_YEAR)

biotic_forestnorm_ic = ee.ImageCollection(
    years_list.map(lambda y: biotic_forestnorm_one_year(ee.Number(y)))
)

biotic_forestnorm = biotic_forestnorm_ic.toBands()
biotic_forestnorm = remove_to_bands_append(biotic_forestnorm)

biotic_forestnorm_masked = biotic_forestnorm.updateMask(
    biotic_forestnorm.gte(0.5)
)

biotic_forestnorm_timeseries_sum = biotic_forestnorm_masked.reduce(ee.Reducer.sum()).rename(
    'biotic_forestnorm_timeseries_sum'
)
biotic_norm_masked_wsum = biotic_forestnorm_masked.addBands(biotic_forestnorm_timeseries_sum)

print("Forest-normalized biotic bands:", biotic_norm_masked_wsum.bandNames().getInfo())


Forest-normalized biotic bands: ['biotic_forestnorm_1997', 'biotic_forestnorm_1998', 'biotic_forestnorm_1999', 'biotic_forestnorm_2000', 'biotic_forestnorm_2001', 'biotic_forestnorm_2002', 'biotic_forestnorm_2003', 'biotic_forestnorm_2004', 'biotic_forestnorm_2005', 'biotic_forestnorm_2006', 'biotic_forestnorm_2007', 'biotic_forestnorm_2008', 'biotic_forestnorm_2009', 'biotic_forestnorm_2010', 'biotic_forestnorm_2011', 'biotic_forestnorm_2012', 'biotic_forestnorm_2013', 'biotic_forestnorm_2014', 'biotic_forestnorm_2015', 'biotic_forestnorm_2016', 'biotic_forestnorm_2017', 'biotic_forestnorm_2018', 'biotic_forestnorm_2019', 'biotic_forestnorm_2020', 'biotic_forestnorm_2021', 'biotic_forestnorm_2022', 'biotic_forestnorm_2023', 'biotic_forestnorm_timeseries_sum']


# EXPORT & HELPERS

In [ ]:
# # EXPORT to asset first, allowing lazy CRS management

# assetID = GEE_PROJECT_DIR + "/biotic_forestnorm"

# # Export to asset
# task_asset = ee.batch.Export.image.toAsset(
#     image=biotic_norm_masked_wsum,
#     description="biotic_forestnorm" + "_toAsset",
#     assetId=assetID,
#     region=western_states.geometry(),
#     scale=1000,
#     maxPixels=1e13
# )

# task_asset.start()



In [11]:
# Re-load and export from pyramid-backed asset, forcing CRS to 5070 but hopefully still allowing sharding

# biotic_asset = ee.Image(assetID).select

# biotic_no_sum = biotic_asset.select(
#     biotic_asset.bandNames().remove("biotic_forestnorm_timeseries_sum")
# )

# Export to Drive; needs tiling due to the forced projection piece
task_drive = ee.batch.Export.image.toDrive(
    image=biotic,
    description="biotic_test" + "_toDrive",
    folder=DRIVE_FOLDER,
    region=western_states.geometry(),
    scale=1000,
    crs=EXPORT_CRS,
    maxPixels=1e13
)

task_drive.start()

In [5]:
import ee

EXPORT_CRS = "EPSG:5070"
SCALE = 1000  # meters

# Robust dissolve of the region
west_geom = western_states.union().geometry()

# Export the image you actually want (includes sum band)
img_to_export = biotic_forestnorm_masked.clip(west_geom)

def bbox_5070(geom):
    """Return bbox (xmin, ymin, xmax, ymax) in EPSG:5070 as client-side floats."""
    b = geom.transform(EXPORT_CRS, 1).bounds(proj=EXPORT_CRS)
    coords = ee.List(ee.List(b.coordinates().get(0)))
    # Rectangle ring: (xmin,ymin)->(xmax,ymin)->(xmax,ymax)->(xmin,ymax)->(xmin,ymin)
    xmin = ee.Number(ee.List(coords.get(0)).get(0))
    ymin = ee.Number(ee.List(coords.get(0)).get(1))
    xmax = ee.Number(ee.List(coords.get(2)).get(0))
    ymax = ee.Number(ee.List(coords.get(2)).get(1))
    return [v.getInfo() for v in [xmin, ymin, xmax, ymax]]

xmin, ymin, xmax, ymax = bbox_5070(west_geom)

xmid = (xmin + xmax) / 2.0
ymid = (ymin + ymax) / 2.0

tiles = [
    ("x00_y00", [xmin, ymin, xmid, ymid]),  # SW
    ("x01_y00", [xmid, ymin, xmax, ymid]),  # SE
    ("x00_y01", [xmin, ymid, xmid, ymax]),  # NW
    ("x01_y01", [xmid, ymid, xmax, ymax]),  # NE
]

tasks = []
for suffix, (x0, y0, x1, y1) in tiles:
    tile_geom = ee.Geometry.Rectangle([x0, y0, x1, y1], proj=EXPORT_CRS, geodesic=False)

    desc = f"biotic_forestnorm_5070_1km_{suffix}"

    task = ee.batch.Export.image.toDrive(
        image=img_to_export,
        description=desc,
        folder=DRIVE_FOLDER,
        fileNamePrefix=desc,
        region=tile_geom,
        crs=EXPORT_CRS,
        scale=SCALE,
        maxPixels=1e13
    )
    task.start()
    tasks.append(task)

print(f"Started {len(tasks)} exports (forced 2x2 tiling).")


Started 4 exports (forced 2x2 tiling).


In [6]:
import ee

EXPORT_CRS = "EPSG:5070"
SCALE = 1000  # meters

west_geom = western_states.union().geometry()

img_to_export = biotic_forestnorm_masked.clip(west_geom)

def bbox_5070(geom):
    b = geom.transform(EXPORT_CRS, 1).bounds(proj=EXPORT_CRS)
    coords = ee.List(ee.List(b.coordinates().get(0)))
    xmin = ee.Number(ee.List(coords.get(0)).get(0))
    ymin = ee.Number(ee.List(coords.get(0)).get(1))
    xmax = ee.Number(ee.List(coords.get(2)).get(0))
    ymax = ee.Number(ee.List(coords.get(2)).get(1))
    return [v.getInfo() for v in [xmin, ymin, xmax, ymax]]

xmin, ymin, xmax, ymax = bbox_5070(west_geom)

xmid = (xmin + xmax) / 2.0
ymid = (ymin + ymax) / 2.0

tiles = [
    ("SW", [xmin, ymin, xmid, ymid]),
    ("SE", [xmid, ymin, xmax, ymid]),
    ("NW", [xmin, ymid, xmid, ymax]),
    ("NE", [xmid, ymid, xmax, ymax]),
]

print("\nFull bbox (5070 meters):")
print(f"xmin={xmin}, ymin={ymin}, xmax={xmax}, ymax={ymax}")

print("\nTile extents:")
for name, (x0, y0, x1, y1) in tiles:
    w = x1 - x0
    h = y1 - y0
    print(f"{name}:")
    print(f"  xmin={x0:.0f}, ymin={y0:.0f}, xmax={x1:.0f}, ymax={y1:.0f}")
    print(f"  width_km={w/1000:.1f}, height_km={h/1000:.1f}")



Full bbox (5070 meters):
xmin=-2361581.263023, ymin=991812.651244, xmax=-504621.14299, ymax=3177422.386741

Tile extents:
SW:
  xmin=-2361581, ymin=991813, xmax=-1433101, ymax=2084618
  width_km=928.5, height_km=1092.8
SE:
  xmin=-1433101, ymin=991813, xmax=-504621, ymax=2084618
  width_km=928.5, height_km=1092.8
NW:
  xmin=-2361581, ymin=2084618, xmax=-1433101, ymax=3177422
  width_km=928.5, height_km=1092.8
NE:
  xmin=-1433101, ymin=2084618, xmax=-504621, ymax=3177422
  width_km=928.5, height_km=1092.8


In [7]:
m = geemap.Map()

for name, (x0,y0,x1,y1) in tiles:
    geom = ee.Geometry.Rectangle([x0,y0,x1,y1], proj=EXPORT_CRS, geodesic=False)
    m.addLayer(geom, {}, name)

m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [17]:
import os
import shutil
import tempfile
import contextlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, List
import time, random

from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload

# GDAL bindings
from osgeo import gdal


def list_all_files_in_folder(
    service,
    folder_id: str,
    description: Optional[str] = None,
    *,
    include_all_drives: bool = True,
    drive_id: Optional[str] = None,
    page_size: int = 1000,
) -> List[dict]:
    """
    List ALL files in a Drive folder, with pagination.
    Filters by description substring and .tif/.TIF extension.
    Returns a list of dicts with at least {'id','name'}.
    """
    # Query: in folder, not trashed, optional name contains 'description',
    # and extension contains '.tif' or '.TIF'
    name_filter = []
    if description:
        name_filter.append(f"(name contains '{description}')")
    ext_filter = "((name contains '.tif') or (name contains '.TIF'))"
    q_parts = [f"'{folder_id}' in parents", "trashed=false", ext_filter]
    if name_filter:
        q_parts.append(" and ".join(name_filter))
    q = " and ".join(q_parts)

    kwargs = dict(
        q=q,
        pageSize=page_size,
        fields="nextPageToken, files(id, name)",
        orderBy="name",
    )
    if include_all_drives:
        kwargs.update(dict(supportsAllDrives=True, includeItemsFromAllDrives=True))
        if drive_id:
            kwargs.update(dict(corpora="drive", driveId=drive_id))
        else:
            kwargs.update(dict(corpora="allDrives"))
    else:
        kwargs.update(dict(corpora="user"))

    files, page_token = [], None
    while True:
        resp = service.files().list(pageToken=page_token, **kwargs).execute()
        files.extend(resp.get("files", []))
        page_token = resp.get("nextPageToken")
        if not page_token:
            break
    return files

def _download_file_from_drive(file_id, file_name, temp_dir, service_account_file):
    try:
        # Authenticate service account
        SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
        credentials = service_account.Credentials.from_service_account_file(
            os.path.abspath(service_account_file), scopes=SCOPES
        )
        service = build('drive', 'v3', credentials=credentials)

        request = service.files().get_media(fileId=file_id)
        local_path = os.path.join(temp_dir, file_name)
        with open(local_path, 'wb') as f:
            downloader = MediaIoBaseDownload(f, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
        time.sleep(random.uniform(0.5, 1.5))  # Random delay
        print(f"Downloaded {file_name}")
        return local_path
    except Exception as e:
        print(f"Error downloading {file_name}: {e}")
        return None


def download_merge_from_drive_tif(
    description: str,
    local_filename: str,
    drive_folder: str,
    service_account_file: str,
    *,
    # Output & mosaic controls
    compress: str = "DEFLATE",          # DEFLATE|LZW|ZSTD|JPEG
    predictor: Optional[int] = None,     # 2 for integers, 3 for floats; None = let GDAL decide
    blocksize: int = 512,                # tile size for output GeoTIFF
    bigtiff: str = "YES",                # YES|IF_SAFER
    dst_srs: Optional[str] = None,       # e.g., "EPSG:5070" (if reprojection needed)
    xres: Optional[float] = None,        # set both xres & yres to enforce pixel size (e.g., 30)
    yres: Optional[float] = None,
    dst_nodata: Optional[float] = None,  # set if you need a specific nodata in output
    # Drive / listing controls
    drive_id: Optional[str] = None,      # if folder is in a Shared Drive, you can pass the Drive ID
    include_all_drives: bool = True,     # search across Shared Drives
    # Behavior controls
    check_existing: bool = True,
    n_workers: int = 4,
    verbose: bool = True,
) -> str:
    """
    Download all Drive .tif/.TIF files whose names contain `description` from `drive_folder`,
    then build a VRT and stream a mosaicked GeoTIFF to `local_filename` (no in-memory merge).
    """

    # --- Fast exit if we already have the target ---
    local_filename = os.path.abspath(local_filename)
    if check_existing and os.path.exists(local_filename):
        if verbose:
            print(f"File already exists: {local_filename}")
        return local_filename

    # --- Temp workspace ---
    temp_dir = tempfile.mkdtemp()
    vrt_path = os.path.join(temp_dir, "mosaic.vrt")
    downloaded_paths: List[str] = []

    # --- Build Drive client ---
    SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
    cred_path = os.path.abspath(service_account_file)
    credentials = service_account.Credentials.from_service_account_file(cred_path, scopes=SCOPES)
    service = build("drive", "v3", credentials=credentials)

    # Convenience flags for Shared Drives
    list_kwargs = dict(
        fields="nextPageToken, files(id, name, parents)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ) if include_all_drives else dict(fields="nextPageToken, files(id, name, parents)")

    try:
        # --- Locate the folder (paginate just in case) ---
        folder_q = (
            f"name='{drive_folder}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        )
        folder_kwargs = dict(
            q=folder_q,
            pageSize=1000,
            corpora=("drive" if drive_id else "allDrives") if include_all_drives else "user",
            driveId=drive_id,
            **list_kwargs
        )

        folders, page_token = [], None
        while True:
            resp = service.files().list(pageToken=page_token, **folder_kwargs).execute()
            folders.extend(resp.get("files", []))
            page_token = resp.get("nextPageToken")
            if not page_token:
                break

        if not folders:
            raise FileNotFoundError(
                f"Folder '{drive_folder}' not found. "
                "Ensure it exists and is shared with the service account "
                f"({credentials.service_account_email}). "
                "If it is on a Shared Drive, also ensure the service account has at least Viewer."
            )
        folder_id = folders[0]["id"]

        # --- List ALL matching .tif files in that folder (paginated) ---
        files = list_all_files_in_folder(
            service,
            folder_id,
            description=description,
            include_all_drives=include_all_drives,
            drive_id=drive_id,
            page_size=1000
        )

        if not files:
            raise FileNotFoundError(
                f"No matching .tif/.TIF files found for '{description}' in '{drive_folder}'. "
                "Check filenames and sharing."
            )

        if verbose:
            print(f"Found {len(files)} file(s). Starting download to {temp_dir} ...")

        # --- Download the files (parallel if requested) ---
        if n_workers <= 1:
            for f in files:
                path = _download_file_from_drive(f["id"], f["name"], temp_dir, service_account_file)
                if path:
                    downloaded_paths.append(path)
        else:
            with ThreadPoolExecutor(max_workers=n_workers) as ex:
                futures = [
                    ex.submit(_download_file_from_drive, f["id"], f["name"], temp_dir, service_account_file)
                    for f in files
                ]
                for fut in as_completed(futures):
                    p = fut.result()
                    if p:
                        downloaded_paths.append(p)

        if not downloaded_paths:
            raise RuntimeError("Downloads finished but no files were saved.")

        if verbose:
            print(f"Building VRT from {len(downloaded_paths)} tile(s) ...")

        # --- Build VRT (lightweight, no data copy) ---
        vrt_options = {}
        if dst_nodata is not None:
            vrt_options.update(dict(srcNodata=dst_nodata, VRTNodata=dst_nodata))

        vrt = gdal.BuildVRT(vrt_path, downloaded_paths, **vrt_options)
        if vrt is None:
            raise RuntimeError("GDAL BuildVRT failed.")
        vrt = None  # flush

        # --- Stream out to GeoTIFF (no giant array in RAM) ---
        co = [
            f"BIGTIFF={bigtiff}",
            "TILED=YES",
            f"COMPRESS={compress.upper()}",
            f"BLOCKXSIZE={int(blocksize)}",
            f"BLOCKYSIZE={int(blocksize)}",
            "OVERVIEWS=AUTO", #pyramids
            "RESAMPLING=AVERAGE",
            "NUM_THREADS=ALL_CPUS"
        ]
        if predictor is not None:
            co.append(f"PREDICTOR={int(predictor)}")

        needs_warp = any(v is not None for v in (dst_srs, xres, yres, dst_nodata))
        if needs_warp:
            if verbose:
                print("Warping (streamed) to target grid...")
            warp_kwargs = dict(
                format="GTiff",
                creationOptions=co,
                multithread=True,
            )
            if dst_srs:
                warp_kwargs["dstSRS"] = dst_srs
            if xres and yres:
                warp_kwargs["xRes"] = float(xres)
                warp_kwargs["yRes"] = float(yres)
                warp_kwargs["targetAlignedPixels"] = True
            if dst_nodata is not None:
                warp_kwargs["dstNodata"] = dst_nodata

            out_ds = gdal.Warp(local_filename, vrt_path, **warp_kwargs)
            if out_ds is None:
                raise RuntimeError("GDAL Warp failed.")
            out_ds = None
        else:
            if verbose:
                print("Translating VRT to tiled, compressed GeoTIFF (streamed)...")
            out_ds = gdal.Translate(local_filename, vrt_path, creationOptions=co)
            if out_ds is None:
                raise RuntimeError("GDAL Translate failed.")
            out_ds = None

        if verbose:
            print(f"Final mosaicked GeoTIFF saved to: {local_filename}")
        return local_filename

    except HttpError as e:
        msg = str(e)
        if "accessNotConfigured" in msg or "has not been used in project" in msg:
            raise RuntimeError(
                "Google Drive API is disabled for this service account’s project.\n"
                "Enable the **Google Drive API** in GCP → APIs & Services → Library, then retry."
            ) from e
        raise
    except KeyboardInterrupt:
        print("Interrupted by user. Cleaning up and exiting.")
        raise
    finally:
        # Best-effort cleanup
        try:
            shutil.rmtree(temp_dir)
            if verbose:
                print(f"Cleaned up temporary directory: {temp_dir}")
        except Exception as cleanup_err:
            print(f"Error during cleanup: {cleanup_err}")


def download_merge_from_drive_cog(
    description: str,
    local_filename: str,
    drive_folder: str,
    service_account_file: str,
    *,
    # Output & mosaic controls
    compress: str = "DEFLATE",          # DEFLATE|LZW|ZSTD|JPEG
    predictor: Optional[int] = None,     # 2 for integers, 3 for floats; None = let GDAL decide
    blocksize: int = 512,                # tile size for output COG (both x and y)
    bigtiff: str = "YES",                # YES|IF_SAFER
    dst_srs: Optional[str] = None,       # e.g., "EPSG:5070" (if reprojection needed)
    xres: Optional[float] = None,        # set both xres & yres to enforce pixel size (e.g., 30)
    yres: Optional[float] = None,
    dst_nodata: Optional[float] = None,  # set if you need a specific nodata in output
    # Drive / listing controls
    drive_id: Optional[str] = None,      # if folder is in a Shared Drive, you can pass the Drive ID
    include_all_drives: bool = True,     # search across Shared Drives
    # Behavior controls
    check_existing: bool = True,
    n_workers: int = 4,
    verbose: bool = True,
) -> str:
    """
    Download all Drive .tif/.TIF files whose names contain `description` from `drive_folder`,
    then build a VRT and stream a mosaicked GeoTIFF to `local_filename` (no in-memory merge).
    """

    # --- Fast exit if we already have the target ---
    local_filename = os.path.abspath(local_filename)
    if check_existing and os.path.exists(local_filename):
        if verbose:
            print(f"File already exists: {local_filename}")
        return local_filename

    # --- Temp workspace ---
    temp_dir = tempfile.mkdtemp()
    vrt_path = os.path.join(temp_dir, "mosaic.vrt")
    downloaded_paths: List[str] = []

    # --- Build Drive client ---
    SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
    cred_path = os.path.abspath(service_account_file)
    credentials = service_account.Credentials.from_service_account_file(cred_path, scopes=SCOPES)
    service = build("drive", "v3", credentials=credentials)

    # Convenience flags for Shared Drives
    list_kwargs = dict(
        fields="nextPageToken, files(id, name, parents)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ) if include_all_drives else dict(fields="nextPageToken, files(id, name, parents)")

    try:
        # --- Locate the folder (paginate just in case) ---
        folder_q = (
            f"name='{drive_folder}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        )
        folder_kwargs = dict(
            q=folder_q,
            pageSize=1000,
            corpora=("drive" if drive_id else "allDrives") if include_all_drives else "user",
            driveId=drive_id,
            **list_kwargs
        )

        folders, page_token = [], None
        while True:
            resp = service.files().list(pageToken=page_token, **folder_kwargs).execute()
            folders.extend(resp.get("files", []))
            page_token = resp.get("nextPageToken")
            if not page_token:
                break

        if not folders:
            raise FileNotFoundError(
                f"Folder '{drive_folder}' not found. "
                "Ensure it exists and is shared with the service account "
                f"({credentials.service_account_email}). "
                "If it is on a Shared Drive, also ensure the service account has at least Viewer."
            )
        folder_id = folders[0]["id"]

        # --- List ALL matching .tif files in that folder (paginated) ---
        files = list_all_files_in_folder(
            service,
            folder_id,
            description=description,
            include_all_drives=include_all_drives,
            drive_id=drive_id,
            page_size=1000
        )

        if not files:
            raise FileNotFoundError(
                f"No matching .tif/.TIF files found for '{description}' in '{drive_folder}'. "
                "Check filenames and sharing."
            )

        if verbose:
            print(f"Found {len(files)} file(s). Starting download to {temp_dir} ...")

        # --- Download the files (parallel if requested) ---
        if n_workers <= 1:
            for f in files:
                path = _download_file_from_drive(f["id"], f["name"], temp_dir, service_account_file)
                if path:
                    downloaded_paths.append(path)
        else:
            with ThreadPoolExecutor(max_workers=n_workers) as ex:
                futures = [
                    ex.submit(_download_file_from_drive, f["id"], f["name"], temp_dir, service_account_file)
                    for f in files
                ]
                for fut in as_completed(futures):
                    p = fut.result()
                    if p:
                        downloaded_paths.append(p)

        if not downloaded_paths:
            raise RuntimeError("Downloads finished but no files were saved.")

        if verbose:
            print(f"Building VRT from {len(downloaded_paths)} tile(s) ...")

        # --- Build VRT (lightweight, no data copy) ---
        vrt_options = {}
        if dst_nodata is not None:
            vrt_options.update(dict(srcNodata=dst_nodata, VRTNodata=dst_nodata))

        vrt = gdal.BuildVRT(vrt_path, downloaded_paths, **vrt_options)
        if vrt is None:
            raise RuntimeError("GDAL BuildVRT failed.")
        vrt = None  # flush

        # --- Stream out to Cloud-Optimized GeoTIFF (COG) ---
        # COG creation options (driver: "COG")
        co_cog = [
            f"COMPRESS={compress.upper()}",     # e.g., DEFLATE|LZW|ZSTD|JPEG
            f"BLOCKSIZE={int(blocksize)}",      # one value for both X/Y, typical 512
            f"NUM_THREADS={'ALL_CPUS'}",
            f"BIGTIFF={bigtiff}",               # YES|IF_SAFER
            "OVERVIEW_RESAMPLING=AVERAGE",      # build internal overviews with this kernel
            # Optional: "SPARSE_OK=YES",       # can help on sparse rasters
        ]
        if predictor is not None:
            co_cog.append(f"PREDICTOR={int(predictor)}")  # 2=int, 3=float (for DEFLATE/LZW)

        needs_warp = any(v is not None for v in (dst_srs, xres, yres, dst_nodata))

        if needs_warp:
            if verbose:
                print("Warping to target grid (VRT) and writing COG...")
            # Warp to a VRT (no data copy), then translate that VRT → COG
            warped_vrt = os.path.join(temp_dir, "warped.vrt")
            warp_kwargs = dict(
                format="VRT",           # keep it as VRT to avoid creating a big intermediate
                multithread=True,
            )
            if dst_srs:
                warp_kwargs["dstSRS"] = dst_srs
            if xres and yres:
                warp_kwargs["xRes"] = float(xres)
                warp_kwargs["yRes"] = float(yres)
                warp_kwargs["targetAlignedPixels"] = True
            if dst_nodata is not None:
                warp_kwargs["dstNodata"] = dst_nodata

            out_warp = gdal.Warp(warped_vrt, vrt_path, **warp_kwargs)
            if out_warp is None:
                raise RuntimeError("GDAL Warp (to VRT) failed.")
            out_warp = None

            # Final: VRT → COG
            out_ds = gdal.Translate(
                local_filename, warped_vrt,
                format="COG",
                creationOptions=co_cog
            )
            if out_ds is None:
                raise RuntimeError("GDAL Translate to COG failed.")
            out_ds = None

        else:
            if verbose:
                print("Translating VRT directly to COG (internal overviews)...")
            out_ds = gdal.Translate(
                local_filename, vrt_path,
                format="COG",
                creationOptions=co_cog
            )
            if out_ds is None:
                raise RuntimeError("GDAL Translate to COG failed.")
            out_ds = None

        if verbose:
            print(f"Final COG saved to: {local_filename}")
        return local_filename

    except HttpError as e:
        msg = str(e)
        if "accessNotConfigured" in msg or "has not been used in project" in msg:
            raise RuntimeError(
                "Google Drive API is disabled for this service account’s project.\n"
                "Enable the **Google Drive API** in GCP → APIs & Services → Library, then retry."
            ) from e
        raise
    except KeyboardInterrupt:
        print("Interrupted by user. Cleaning up and exiting.")
        raise
    finally:
        # Best-effort cleanup
        try:
            shutil.rmtree(temp_dir)
            if verbose:
                print(f"Cleaned up temporary directory: {temp_dir}")
        except Exception as cleanup_err:
            print(f"Error during cleanup: {cleanup_err}")

In [ ]:
download_merge_from_drive_tif(
    description="biotic_forestnorm",
    local_filename=os.path.join(DIR_DERIVED, "biotic_forestnorm.tif"),
    drive_folder=DRIVE_FOLDER,
    service_account_file=SERVICE_FILE,
    compress="DEFLATE",         # same as 'deflate', case-insensitive
    predictor=3,                # 3 = best for float32 data (like indices or continuous values)
    blocksize=512,              # tile size; 256–512 is good default
    bigtiff="YES",              # ensures large mosaics don’t overflow 4 GB
    check_existing=True,
    n_workers=4,                # parallel downloads from Drive
    include_all_drives=True,    # safer if GEE_Exports is on a Shared Drive
    verbose=True
)